# Applied AI SampleID — Sony model

This notebook lets you run Sony's pretrained SampleID model entirely in Google Colab.

**Workflow:** upload the original/source song → upload the sampled/derivative song → run analysis → review candidate matching timestamps → download CSV.

> These are candidate matching windows, not final legal-grade sample boundaries.

In [ ]:
# 1) Install Sony SampleID and audio dependencies
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!rm -rf /content/sampleid
!git clone -q https://github.com/sony/sampleid.git /content/sampleid
!pip -q install -e /content/sampleid librosa pandas soundfile tqdm

print('Setup complete.')

In [ ]:
# 2) Download Sony's pretrained checkpoint
from pathlib import Path
import os, zipfile, requests

CKPT_PATH = Path('/content/sampleid-best.ckpt')
CKPT_URLS = [
    'https://zenodo.org/api/records/17413869/files/sampleid-best.ckpt/content',
    'https://zenodo.org/records/17413869/files/sampleid-best.ckpt?download=1',
]

def valid_checkpoint(path):
    return path.exists() and path.stat().st_size > 10_000_000 and zipfile.is_zipfile(path)

if not valid_checkpoint(CKPT_PATH):
    CKPT_PATH.unlink(missing_ok=True)
    last_error = None
    for url in CKPT_URLS:
        try:
            print('Downloading model...')
            with requests.get(url, stream=True, timeout=180, headers={'User-Agent':'Mozilla/5.0'}) as r:
                r.raise_for_status()
                with open(CKPT_PATH, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        if chunk:
                            f.write(chunk)
            if valid_checkpoint(CKPT_PATH):
                break
        except Exception as e:
            last_error = e
            CKPT_PATH.unlink(missing_ok=True)
    if not valid_checkpoint(CKPT_PATH):
        raise RuntimeError(f'Could not download checkpoint: {last_error}')

print(f'Checkpoint ready: {CKPT_PATH} ({CKPT_PATH.stat().st_size/1e6:.1f} MB)')

In [ ]:
# 3) Load the Sony model
import sys, torch
sys.path.insert(0, '/content/sampleid')
sys.path.insert(0, '/content/sampleid/sampleid')
from sampleid import SampleID

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SampleID.load_checkpoint(ckpt_path=str(CKPT_PATH), device=device)
model.eval()
print('Model loaded on', device)

In [ ]:
# 4) Upload the ORIGINAL / SOURCE song
from google.colab import files
print('Choose the original/source song:')
uploaded_original = files.upload()
original_path = next(iter(uploaded_original.keys()))
print('Original:', original_path)

In [ ]:
# 5) Upload the SAMPLED / DERIVATIVE song
print('Choose the sampled/derivative song:')
uploaded_derivative = files.upload()
derivative_path = next(iter(uploaded_derivative.keys()))
print('Derivative:', derivative_path)

In [ ]:
# 6) Analyze the two songs
import librosa, numpy as np, pandas as pd

SR = 16_000
CHUNK_SECONDS = 5.0
HOP_SECONDS = 2.5
MIN_MATCH_THRESHOLD = 0.60  # change this if you want a stricter/looser filter
BATCH_SIZE = 32

def chunk_audio(path, chunk_seconds, hop_seconds):
    y, _ = librosa.load(path, sr=SR, mono=True)
    chunk_n = int(chunk_seconds * SR)
    hop_n = int(hop_seconds * SR)
    chunks, spans = [], []
    if len(y) <= chunk_n:
        padded = np.pad(y, (0, max(0, chunk_n-len(y))))[:chunk_n].astype(np.float32)
        return np.stack([padded]), [(0.0, len(y)/SR)]
    for start_n in range(0, len(y)-chunk_n+1, hop_n):
        end_n = start_n + chunk_n
        chunks.append(y[start_n:end_n].astype(np.float32))
        spans.append((start_n/SR, end_n/SR))
    return np.stack(chunks), spans

def embed(audio_chunks):
    outs = []
    for i in range(0, len(audio_chunks), BATCH_SIZE):
        x = torch.from_numpy(audio_chunks[i:i+BATCH_SIZE]).to(device)
        with torch.inference_mode():
            z = model(x, audio=True).squeeze(1)
        z = torch.nn.functional.normalize(z, dim=-1)
        outs.append(z.cpu().numpy())
    return np.concatenate(outs, axis=0)

print('Chunking audio...')
o_audio, o_spans = chunk_audio(original_path, CHUNK_SECONDS, HOP_SECONDS)
d_audio, d_spans = chunk_audio(derivative_path, CHUNK_SECONDS, HOP_SECONDS)
print(f'Original chunks: {len(o_audio)} | Derivative chunks: {len(d_audio)}')

print('Computing embeddings...')
o = embed(o_audio)
d = embed(d_audio)
sim = d @ o.T

rows = []
for di in range(sim.shape[0]):
    for oi in range(sim.shape[1]):
        score = float(sim[di, oi])
        if score >= MIN_MATCH_THRESHOLD:
            rows.append({
                'similarity': round(score, 4),
                'original_start_s': round(o_spans[oi][0], 2),
                'original_end_s': round(o_spans[oi][1], 2),
                'derivative_start_s': round(d_spans[di][0], 2),
                'derivative_end_s': round(d_spans[di][1], 2),
            })

df = pd.DataFrame(rows)
if len(df):
    df = df.sort_values('similarity', ascending=False).reset_index(drop=True)
    display(df.head(200))
    print(f'{len(df)} candidate window pair(s) met threshold {MIN_MATCH_THRESHOLD:.2f}.')
else:
    print(f'No candidate matches met threshold {MIN_MATCH_THRESHOLD:.2f}.')

In [ ]:
# 7) Download the results CSV
if len(df):
    out = 'sampleid_matches.csv'
    df.to_csv(out, index=False)
    files.download(out)
else:
    print('No results to download.')

## How to read the result

A row such as **original 42.5–47.5 s ↔ derivative 70.0–75.0 s** means Sony's model considers those two 5-second windows similar enough to meet the selected threshold.

The **similarity** number is a model similarity score, not a probability that sampling occurred. The threshold still needs to be calibrated on known project examples.

This is Step 1 candidate localization. A separate refinement step is still needed for exact start/end boundaries and Nature of Use measurements.